# Evaluación final E2 — 50 checkpoints cerrados
Sólo inferencia de test, sin entrenamiento ni ajuste. El usuario autorizó este cierre tras revisar validación.
Selecciona **GPU**. Conserva en Drive `tesis_imo_e2_arquitecturas_reales_v2` con las 50 carpetas originales.
Sube **paquete_test_e2_cerrado.zip** al panel de archivos o a la raíz de Drive. Este ZIP contiene el test y código fijado, no los checkpoints.
No ejecutar a la vez el notebook de entrenamiento. El test tuvo exposición previa en E1; se declara esa limitación.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json, hashlib, zipfile
from pathlib import Path
def run(command, check=True):
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=os.environ.copy()) as process:
        for line in process.stdout: print(line,end='',flush=True)
        code=process.wait()
    if check and code: raise RuntimeError(f'Comando fallido ({code}); revisa el error anterior: {command}')
RUNS=Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')
OUTPUT=Path('/content/drive/MyDrive/tesis_imo_e2_test_final_v1')
assert RUNS.is_dir(), 'Falta la carpeta de checkpoints en Drive'
OUTPUT.mkdir(parents=True,exist_ok=True)
run(['mountpoint','-q','/content/drive'])
run(['nvidia-smi'])


In [ ]:
EXPECTED_SHA256='ce8b6ecc48df618e59ed4a4de86c693731e4434857830921f40eeddd8ef6076c'
ZIP=next((p for p in [Path('/content/paquete_test_e2_cerrado.zip'),Path('/content/drive/MyDrive/paquete_test_e2_cerrado.zip')] if p.exists()),None)
if ZIP is None: raise FileNotFoundError('Sube paquete_test_e2_cerrado.zip y repite esta celda')
def sha_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
assert sha_file(ZIP)==EXPECTED_SHA256,'ZIP distinto del protocolo cerrado'
BASE=Path('/content/imo_e2_test_final');BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for name in z.namelist():
        assert (BASE/name).resolve().is_relative_to(BASE),'Ruta inválida'
    z.extractall(BASE)
for name,expected in json.loads((BASE/'PACKAGE_MANIFEST.json').read_text()).items():
    assert sha_file(BASE/name)==expected,name
PROJECT=BASE/'tesis_imo_para_codex'
print('Paquete verificado. Checkpoints se leerán de Drive.')


## Entorno completo — también funciona después de reiniciar la sesión
Incluye setuptools, cuya ausencia causó el error anterior. No cambia los archivos de entrenamiento.

In [ ]:
run([sys.executable,'-m','pip','install','uv==0.6.9'])
ENV=Path('/content/imo-e2-env');PY=str(ENV/'bin/python')
if not Path(PY).exists():run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)])
def install(*items):run([sys.executable,'-m','uv','pip','install','--python',PY,*items])
install('setuptools==75.8.0')
install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
install('-r',str(PROJECT/'requirements-e2.txt'))
install('transformers==4.46.3')
install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
site=ENV/'lib/python3.11/site-packages'
dirs=[site/'torch/lib',*sorted((site/'nvidia').glob('*/lib'))]
paths=[str(p) for p in dirs if p.is_dir()]+os.environ.get('LD_LIBRARY_PATH','').split(':')
os.environ['LD_LIBRARY_PATH']=':'.join(dict.fromkeys(p for p in paths if p))
run([PY,'-u','-c','import torch,torchvision,triton,mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'])
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)


## Evaluar o continuar
Verifica los 50 checkpoints antes de inferencia. Muestra progreso cada 50 imágenes y guarda cada corrida completa. Si se interrumpe una corrida, sólo esa corrida se repite; las completadas se verifican y omiten.
No hay optimizador, backward, búsqueda de umbral ni selección de checkpoint con test. Usa las mismas cinco semillas y umbral 0,5.


In [ ]:
run([PY,'-u',str(PROJECT/'scripts/evaluar_e2_cerrado.py'),'--runs',str(RUNS),'--output',str(OUTPUT)])


In [ ]:
summary=json.loads((OUTPUT/'RESUMEN_TEST.json').read_text())
for r in summary['summary']:
    print(r['model'], 'Dice:',round(r['Dice_mean'],4),'DE semillas:',round(r['Dice_sd_seeds'],4),'IC:',r['Dice_CI95_groups_fixed_seeds'])
print((OUTPUT/'INFORME_TEST.md').read_text())


## Descargar resultados para revisión
Incluye métricas por imagen/grupo/semilla, intervalos, protocolo y máscaras de la semilla 11. No incluye pesos ni imágenes originales.

In [ ]:
EXPORT=Path('/content/resultados_test_e2_final.zip')
with zipfile.ZipFile(EXPORT,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file() and not p.name.endswith('.tmp'):z.write(p,p.relative_to(OUTPUT))
from google.colab import files
files.download(str(EXPORT))
